# Risk scoring and explainability

Up to this point we've evaluated models in isolation. A risk scoring framework is the layer that turns model output into operational decisions: scores get bucketed into bands, bands get mapped to actions (allow / monitor / review / block), and individual high-risk transactions get explanations that an analyst can act on.

This notebook combines the supervised XGBoost model from `03_model_baseline.ipynb` with the Isolation Forest from `04_anomaly_detection.ipynb` into a single risk score, defines bands, demonstrates per-transaction explanations with SHAP, and discusses what a real fintech deployment would still need beyond what is implemented here.


In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

PROCESSED_DIR = Path("..") / "data" / "processed"
MODELS_DIR = Path("..") / "src" / "models"
FIGURES_DIR = Path("..") / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")


In [4]:
test = pd.read_csv(PROCESSED_DIR / "test.csv")
X_test = test.drop(columns=["isFraud"])
y_test = test["isFraud"]

xgb_model = joblib.load(MODELS_DIR / "xgb_baseline.pkl")
iso_model = joblib.load(MODELS_DIR / "isolation_forest.pkl")

print(f"Test set: {X_test.shape}, fraud rate {y_test.mean():.4%}")


EOFError: 

## 1. Combining supervised and anomaly scores

The two models produce scores on different scales. XGBoost outputs a probability in [0, 1]. Isolation Forest produces a real-valued anomaly score whose range depends on the data. Min-max normalisation puts the anomaly score on [0, 1] so the two can be combined linearly.

Two design choices to be explicit about:

- **The min-max normalisation uses the test-set range.** In a real deployment this should be fit on the training distribution and saved alongside the model — otherwise an unusual day's traffic shifts the scale of every alert. We use the test range here because it's a self-contained illustration.
- **The combination weight is fixed, not tuned.** We use 0.7 supervised + 0.3 anomaly. The weight is a *risk policy decision*, not a hyperparameter to optimise. A risk team that prioritises catching known fraud sets it higher on supervised; a team worried about novel fraud patterns sets it higher on anomaly. Tuning the weight against test-set PR-AUC would defeat the purpose, since the anomaly detector exists precisely to catch what the labels don't show.


In [3]:
# Supervised score: probability of fraud from XGBoost.
xgb_score = xgb_model.predict_proba(X_test)[:, 1]

# Anomaly score: negative decision_function (higher = more anomalous).
iso_raw = -iso_model.decision_function(X_test)
iso_score = (iso_raw - iso_raw.min()) / (iso_raw.max() - iso_raw.min())

# Combined risk score on [0, 1]. The weights reflect a risk policy choice
# rather than an optimisation outcome — see the markdown above.
W_SUPERVISED = 0.7
W_ANOMALY = 0.3
risk_score = W_SUPERVISED * xgb_score + W_ANOMALY * iso_score

scored = pd.DataFrame({
    "xgb_score":  xgb_score,
    "iso_score":  iso_score,
    "risk_score": risk_score,
    "isFraud":    y_test.values,
}, index=y_test.index)
scored.head()


NameError: name 'xgb_model' is not defined